# Full-frame performance measurements

The primary source is a centered AB 20 flat-$f_
u$ continuum. Every arm and the full configured wavelength range are included. Setup, observation and readout are timed separately. `profile=True` records Yappi wall-clock profiles, including profiler overhead; set it to `False` for interactive timings. For independent peak-memory measurements, restart this notebook's kernel between the ZShooter and X-shooter sections and rerun the imports.

The saved CSV retains the earlier baseline and intermediate measurements. `shared_final` uses the adopted branches. The historical X-shooter process peak follows earlier ZShooter runs and is cumulative. Sky downloads affect setup time; the X-shooter section uses an optional local ESO cache with the displayed fixed query parameters. The [comparison notebook](../../notebooks/02_Flat_AB_Comparison.ipynb) separately times its common-sky observations.


In [1]:
from pathlib import Path
from time import perf_counter
import resource
import sys
import numpy as np
import pandas as pd
import yappi
import scopesim as sim
from scopesim.source.source_templates import ab_spectrum
from zs_scopesim_tools.helpers import configure_irdb_path, warning_prevent_sync_alt_ra_dec, disable_scopesim_progress_bars, set_scopesim_log_level

workspace = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'env/zshooter-stack.toml').exists())
irdb = Path(configure_irdb_path())
set_scopesim_log_level('warning')
output = workspace / 'validation/xshooter/data'
measurements = pd.read_csv(output / 'full_frame_timings.csv').to_dict('records')
profile = False
source = sim.Source(x=[0.], y=[0.], ref=[0], weight=[1.], spectra=[ab_spectrum(mag=20)])
print(sys.executable, sim.__version__, sim.__file__)

/Users/jibailey/miniforge3/envs/zssim/bin/python 0.12.1.dev6 /Users/jibailey/src/scopesim/scopesim/__init__.py


In [2]:
instrument = 'ZShooter_v2'
label = 'double_AO033_cache_audit'
properties = {'!OBS.airmass': 1.1, '!OBS.seeing': .65, '!OBS.brightness': 'dark', '!OBS.mjdobs': 60310.25}
properties.update({'!INST.vis_curr_slit': .33, '!INST.nir_curr_slit': .33, '!INST.include_diffuse_emissivity': False, '!ATMO.skycalc_timeout': 60.})
for arm in ('blue', 'green', 'red', 'yj', 'h', 'k'):
    properties[f'!OBS.dit_{arm}'] = 1800.
    properties[f'!OBS.ndit_{arm}'] = 2
started = perf_counter()
cmds = sim.UserCommands(use_instrument=instrument, set_modes=['SPEC'], properties=properties)
warning_prevent_sync_alt_ra_dec(cmds)
train = sim.OpticalTrain(cmds)
setup_s = perf_counter() - started
started = perf_counter()
train.update(migrate_cmds=True)
update_s = perf_counter() - started
train['seeing_psf'].meta['enable_ao'] = True
print(f'{instrument}: setup {setup_s:.2f} s, explicit update {update_s:.2f} s')

py.warnings - WARNING: /Users/jibailey/src/scopesim/scopesim/effects/spectral_trace_list.py:170: DeprecationWarning: The fov_grid method is deprecated and will be removed in a future release.
  fov = thetrace.fov_grid()

py.warnings - WARNING: /Users/jibailey/src/scopesim/scopesim/effects/spectral_trace_list.py:225: DeprecationWarning: The fov_grid method is deprecated and will be removed in a future release.
  vol = spt.fov_grid()

py.warnings - WARNING: /Users/jibailey/src/scopesim/scopesim/effects/spectral_trace_list.py:225: DeprecationWarning: The fov_grid method is deprecated and will be removed in a future release.
  vol = spt.fov_grid()

ZShooter_v2: setup 10.28 s, explicit update 0.88 s


In [3]:
yappi.clear_stats()
yappi.set_clock_type('wall')
if profile:
    yappi.start()
started = perf_counter()
with disable_scopesim_progress_bars(True):
    for plane in train.image_planes:
        plane.data.fill(0)
    train.observe(source, update=False)
observed = perf_counter()
readouts = train.readout()
finished = perf_counter()
if profile:
    yappi.stop()
    stats = yappi.get_func_stats()
    stats.sort('ttot', 'desc')
    with (output / f'{label}_{instrument}_yappi.txt').open('w') as stream:
        stats.print_all(out=stream, columns={0: ('name', 140), 1: ('ncall', 12), 2: ('tsub', 12), 3: ('ttot', 12)})
measurements.append({'instrument': instrument, 'label': label, 'profiled': profile, 'setup_s': setup_s, 'update_s': update_s, 'observe_s': observed - started, 'readout_s': finished - observed, 'peak_process_gib': resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024**3})
pd.DataFrame(measurements).to_csv(output / 'full_frame_timings.csv', index=False)
print([(h[1].data.shape, np.isfinite(h[1].data).all()) for h in readouts])
display(pd.DataFrame(measurements))

[((4096, 2048), np.True_), ((4096, 2048), np.True_), ((4096, 2048), np.True_), ((2048, 2048), np.True_), ((2048, 2048), np.True_), ((2048, 2048), np.True_)]


,instrument,label,profiled,setup_s,observe_s,readout_s,peak_process_gib,update_s
0,ZShooter_v2,before_upstream,True,64.076229,70.069178,1.528787,6.245590,NaN
1,ZShooter_v2,before_upstream,True,64.076229,68.716845,1.551403,7.379410,NaN
2,XSHOOTER,before_upstream,True,1.132821,590.347333,1.146228,35.741592,NaN
3,ZShooter_v2,shared_spatial_first,True,8.305929,74.193853,1.656233,5.894028,NaN
4,XSHOOTER,shared_cached_shifts,True,1.267620,97.764302,0.887620,9.866638,NaN
5,ZShooter_v2,shared_final,True,8.547715,60.892919,1.680166,5.118393,NaN
6,XSHOOTER,shared_final,True,1.373102,74.404743,0.876663,9.322647,NaN
7,ZShooter_v2,repaired_baseline,True,70.700472,71.062688,1.663177,7.080276,NaN
8,ZShooter_v2,shared_finite_cells_AO033,True,9.174564,176.528393,1.707825,18.196976,NaN
9,ZShooter_v2,shared_finite_cells_AO033_fast,False,8.803861,118.712173,1.572954,18.318146,NaN


In [2]:
instrument = 'XSHOOTER'
label = 'double_faint_cache_audit'
properties = {'!OBS.airmass': 1.1, '!OBS.seeing': .65, '!OBS.brightness': 'dark', '!OBS.mjdobs': 60310.25, '!ATMO.skycalc_timeout': 60.}
properties.update({'!DET.biny_uvb': 2, '!DET.biny_vis': 2})
for arm in ('uvb', 'vis', 'nir'):
    properties[f'!OBS.dit_{arm}'] = 300. if arm == 'nir' else 1800.
    properties[f'!OBS.ndit_{arm}'] = 12 if arm == 'nir' else 2
started = perf_counter()
cmds = sim.UserCommands(use_instrument=instrument, set_modes=['SPEC'], properties=properties)
warning_prevent_sync_alt_ra_dec(cmds)
train = sim.OpticalTrain(cmds)
setup_s = perf_counter() - started
started = perf_counter()
train.update(migrate_cmds=True)
update_s = perf_counter() - started
print(f'{instrument}: setup {setup_s:.2f} s, explicit update {update_s:.2f} s')

py.warnings - WARNING: /Users/jibailey/src/scopesim/scopesim/effects/spectral_trace_list.py:170: DeprecationWarning: The fov_grid method is deprecated and will be removed in a future release.
  fov = thetrace.fov_grid()

XSHOOTER: setup 2.64 s, explicit update 0.03 s


In [3]:
yappi.clear_stats()
yappi.set_clock_type('wall')
if profile:
    yappi.start()
started = perf_counter()
with disable_scopesim_progress_bars(True):
    for plane in train.image_planes:
        plane.data.fill(0)
    train.observe(source, update=False)
observed = perf_counter()
readouts = train.readout()
finished = perf_counter()
if profile:
    yappi.stop()
    stats = yappi.get_func_stats()
    stats.sort('ttot', 'desc')
    with (output / f'{label}_{instrument}_yappi.txt').open('w') as stream:
        stats.print_all(out=stream, columns={0: ('name', 140), 1: ('ncall', 12), 2: ('tsub', 12), 3: ('ttot', 12)})
measurements.append({'instrument': instrument, 'label': label, 'profiled': profile, 'setup_s': setup_s, 'update_s': update_s, 'observe_s': observed - started, 'readout_s': finished - observed, 'peak_process_gib': resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024**3})
pd.DataFrame(measurements).to_csv(output / 'full_frame_timings.csv', index=False)
print([(h[1].data.shape, np.isfinite(h[1].data).all()) for h in readouts])
display(pd.DataFrame(measurements))

py.warnings - WARNING: /Users/jibailey/src/scopesim/scopesim/effects/spectral_trace_list.py:225: DeprecationWarning: The fov_grid method is deprecated and will be removed in a future release.
  vol = spt.fov_grid()

[((1500, 2048), np.True_), ((2000, 2048), np.True_), ((1100, 2048), np.True_)]


,instrument,label,profiled,setup_s,observe_s,readout_s,peak_process_gib,update_s
0,ZShooter_v2,before_upstream,True,64.076229,70.069178,1.528787,6.245590,NaN
1,ZShooter_v2,before_upstream,True,64.076229,68.716845,1.551403,7.379410,NaN
2,XSHOOTER,before_upstream,True,1.132821,590.347333,1.146228,35.741592,NaN
3,ZShooter_v2,shared_spatial_first,True,8.305929,74.193853,1.656233,5.894028,NaN
4,XSHOOTER,shared_cached_shifts,True,1.267620,97.764302,0.887620,9.866638,NaN
5,ZShooter_v2,shared_final,True,8.547715,60.892919,1.680166,5.118393,NaN
6,XSHOOTER,shared_final,True,1.373102,74.404743,0.876663,9.322647,NaN
7,ZShooter_v2,repaired_baseline,True,70.700472,71.062688,1.663177,7.080276,NaN
8,ZShooter_v2,shared_finite_cells_AO033,True,9.174564,176.528393,1.707825,18.196976,NaN
9,ZShooter_v2,shared_finite_cells_AO033_fast,False,8.803861,118.712173,1.572954,18.318146,NaN
